# 第3部分：单调性检验

**目的：** 验证特征和坏账率是否有"越大越危险"或"越小越危险"的稳定关系

## 为什么需要单调性？

举例：如果"借款次数"这个特征：
- 借1-3次：坏账率5%
- 借4-6次：坏账率3%  ← 突然降了？
- 借7-10次：坏账率8%

这就是不单调——不能简单说"借越多越危险"，模型也很难学到稳定规律。

单调的特征更可靠、更好解释、上线后更稳定。

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

## 步骤1：Cochran-Armitage趋势检验

统计学方法，检验"分箱从左到右，坏账率是否有显著趋势"

- p < 0.05 → 有显著趋势（好！说明特征有方向性）
- p >= 0.05 → 没有显著趋势（不好，特征可能没用）

In [ ]:
def cochran_armitage_test(table):
    """
    Cochran-Armitage趋势检验

    参数：
        table: 2行 x K列的数组
            第0行 = 每箱的好人数
            第1行 = 每箱的坏人数

    返回：
        z_stat: Z统计量（绝对值越大趋势越明显）
        p_value: p值（<0.05说明趋势显著）

    原理：
        给每个箱一个"分数"(0,1,2,3,...)
        检验坏账率和分数之间是否有线性趋势
    """
    table = np.array(table, dtype=float)
    n_groups = table.shape[1]

    # 每箱的总人数和坏人数
    col_totals = table.sum(axis=0)  # 每箱总人数
    row_totals = table.sum(axis=1)  # 好人总数、坏人总数
    N = table.sum()                  # 总样本数

    # 给每个箱的"分数"：0, 1, 2, ..., K-1
    scores = np.arange(n_groups)

    # 计算统计量
    p_hat = row_totals[1] / N  # 整体坏账率

    # 加权分数
    T = np.sum(scores * table[1, :])  # 坏人的加权分数和
    T_expected = p_hat * np.sum(scores * col_totals)

    # 方差
    var_T = p_hat * (1 - p_hat) * (
        np.sum(scores**2 * col_totals) -
        (np.sum(scores * col_totals))**2 / N
    )

    if var_T <= 0:
        return 0, 1.0

    z_stat = (T - T_expected) / np.sqrt(var_T)

    from scipy.stats import norm
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))  # 双侧检验

    return z_stat, p_value

## 步骤2：分析单个变量的单调性

In [ ]:
def analyze_variable(df, var_name, label_col='dob4_ever10_flg', n_bins=5):
    """
    综合分析一个变量的单调性

    返回：
        dict: 包含趋势方向、Spearman相关、CA检验p值、违反率
    """
    # 等频分箱
    try:
        df['_bin'] = pd.qcut(df[var_name], q=n_bins, duplicates='drop')
    except Exception:
        return None

    # 计算每箱坏账率
    bin_stats = df.groupby('_bin')[label_col].agg(['mean', 'count', 'sum'])
    bin_stats.columns = ['bad_rate', 'total', 'bads']
    bin_stats = bin_stats.sort_index()

    if len(bin_stats) < 3:
        df.drop('_bin', axis=1, inplace=True)
        return None

    bad_rates = bin_stats['bad_rate'].values

    # 1. Spearman相关系数（排序相关）
    ranks = np.arange(len(bad_rates))
    corr, _ = spearmanr(ranks, bad_rates)

    # 2. 判断趋势方向
    if corr > 0.3:
        direction = 'increasing'  # 越大越危险
    elif corr < -0.3:
        direction = 'decreasing'  # 越小越危险
    else:
        direction = 'no_trend'    # 没有明确方向

    # 3. 单调性违反率
    diffs = np.diff(bad_rates)
    if direction == 'increasing':
        violations = np.sum(diffs < 0) / len(diffs)
    elif direction == 'decreasing':
        violations = np.sum(diffs > 0) / len(diffs)
    else:
        violations = 1.0

    # 4. Cochran-Armitage检验
    goods = bin_stats['total'].values - bin_stats['bads'].values
    bads = bin_stats['bads'].values
    ca_table = np.array([goods, bads])
    z_stat, p_value = cochran_armitage_test(ca_table)

    df.drop('_bin', axis=1, inplace=True)

    return {
        'variable': var_name,
        'direction': direction,
        'spearman_corr': corr,
        'violation_ratio': violations,
        'ca_z_stat': z_stat,
        'ca_p_value': p_value,
        'is_monotonic': violations <= 0.2 and p_value < 0.05
    }

## 步骤3：批量分析所有变量

In [ ]:
def batch_analyze(df, feature_list, label_col='dob4_ever10_flg'):
    """批量分析特征单调性"""
    results = []
    for var in feature_list:
        result = analyze_variable(df, var, label_col)
        if result:
            results.append(result)

    result_df = pd.DataFrame(results)

    # 统计
    mono_count = result_df['is_monotonic'].sum()
    print(f'单调特征数: {mono_count}/{len(result_df)}')
    print(f'单调比例: {mono_count/len(result_df):.1%}')

    return result_df

# 使用
mono_results = batch_analyze(df, selected_features)

# 只保留单调的特征
monotonic_features = mono_results[mono_results['is_monotonic']]['variable'].tolist()
print(f'\n保留 {len(monotonic_features)} 个单调特征进入LGBM训练')

---
### 面试考点

| 问题 | 答案 |
|------|------|
| 为什么要求单调性？ | 单调=可解释+稳定，非单调的特征上线后容易出问题 |
| violation_ratio是什么？ | 坏账率"倒退"的箱数占比，<20%算单调 |
| Cochran-Armitage检验是什么？ | 检验分箱从左到右是否有显著的线性趋势 |
| Spearman vs Pearson区别？ | Spearman看排序相关（单调就行），Pearson看线性相关 |
| 不单调的特征一定不能用吗？ | 不一定，可以做WOE编码强制单调化，但需要业务判断 |